# TTAP–MDP: validación del modelo determinista

## Parte 1 — Entidades

Este notebook inicia la segunda parte de la investigación. En esta primera versión no hay incertidumbre, fallas ni aprendizaje por refuerzo. El objetivo es verificar que nodos, recursos, helicópteros, tareas y estados iniciales estén definidos de forma coherente con el TTAP determinista.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ttap_mdp import (
    CoordinateSystem,
    Helicopter,
    Node,
    ResourceVector,
    Task,
    initial_helicopter_state,
    initial_task_state,
)

### 1. Nodos

La clase `Node` admite coordenadas cartesianas para experimentos sintéticos y coordenadas geográficas para el caso Talcahuano.

In [ ]:
base = Node(
    node_id="BASE",
    name="Talcahuano Base",
    first_coordinate=-36.6911135,
    second_coordinate=-73.04786,
    coordinate_system=CoordinateSystem.GEOGRAPHIC_DEGREES,
    is_base=True,
)

node_a = Node(
    node_id="A",
    name="Demand node A",
    first_coordinate=-36.7006254,
    second_coordinate=-73.1214138,
    coordinate_system=CoordinateSystem.GEOGRAPHIC_DEGREES,
)

base, node_a

### 2. Helicóptero y tarea

Las especificaciones inmutables se mantienen separadas de los estados que cambiarán durante la simulación.

In [ ]:
h1 = Helicopter(
    helicopter_id="H1",
    model="H125_light",
    initial_node_id=base.node_id,
    capacity=ResourceVector(cargo=300, medical=1, personnel=2),
    speed_kmh=190,
    visual_capable=True,
)

task_1 = Task(
    task_id="T1",
    node_id=node_a.node_id,
    requirements=ResourceVector(cargo=100, medical=1),
    optimal_time=10,
    effective_time=30,
    ineffective_time=90,
    priority_weight=2.0,
    service_time=6,
    release_time=0,
    requires_visual_capability=True,
)

h1, task_1

### 3. Compatibilidad y consumo de recursos

In [ ]:
assert h1.is_compatible_with(task_1)

remaining = h1.capacity.consume(task_1.requirements)

print("Priority class:", task_1.priority_class.value)
print("Initial resources:", h1.capacity.as_tuple())
print("Task requirements:", task_1.requirements.as_tuple())
print("Remaining resources:", remaining.as_tuple())

### 4. Estados iniciales deterministas

En la etapa determinista, cada helicóptero comienza disponible en la base y con sus capacidades completas. Una tarea con `release_time = 0` comienza pendiente.

In [ ]:
h1_state = initial_helicopter_state(h1)
task_1_state = initial_task_state(task_1, current_time=0)

assert h1_state.node_id == base.node_id
assert h1_state.remaining_resources == h1.capacity
assert task_1_state.status.value == "pending"

h1_state, task_1_state

## Resultado de esta etapa

Las entidades y sus invariantes básicas quedaron validadas. El siguiente bloque incorporará `scenario.py`: una colección coherente de nodos, helicópteros y tareas, junto con validaciones de identificadores, referencias y una única base.